# 02: Prepare Benchmarks

**Run once.** Pre-converts ScienceQA + Mind2Web into our unified JSONL format,
drops images to disk, and bundles the in-repo synthetic sample. The output of
this notebook becomes the `gvla-data` Kaggle Dataset that every eval notebook
mounts.

## Prerequisites

1. The `grounded_vla` repo is reachable (we clone from GitHub here; substitute
   your fork's URL).
2. Internet = `On` and CPU runtime is fine for this notebook (no GPU needed).

Approximate runtime: 10–20 minutes depending on the slice size.

In [ ]:
# Clone repo (skip if already attached as a Kaggle Dataset).
import subprocess, os
REPO_URL = 'https://github.com/<your-org>/grounded_vla.git'
if not os.path.exists('/kaggle/working/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/kaggle/working/grounded_vla'], check=True)
%cd /kaggle/working/grounded_vla

In [ ]:
!pip install -q -e .[data]

In [ ]:
# Set HF token if your dataset slice is gated (most are not).
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
    print('HF auth OK')
except Exception as e:
    print('Skipping HF auth (probably fine):', e)

In [ ]:
# ScienceQA (test split, image-bearing rows only). 500 tasks ~= 12 minutes.
OUT = '/kaggle/working/gvla-data/scienceqa'
!python scripts/prepare_scienceqa.py --split test --out-dir {OUT} --limit 500
!ls {OUT} && wc -l {OUT}/test.jsonl

In [ ]:
# Synthetic sample (already in the repo, just copy into the data folder).
import shutil
from pathlib import Path
src = Path('data/samples')
dst = Path('/kaggle/working/gvla-data/synthetic_sample')
dst.mkdir(parents=True, exist_ok=True)
shutil.copytree(src / 'images', dst / 'images', dirs_exist_ok=True)
shutil.copy(src / 'synthetic_sample.jsonl', dst / 'synthetic_sample.jsonl')
print('synthetic sample @', dst)

In [ ]:
# Final tree.
import subprocess
subprocess.run(['du', '-sh', '/kaggle/working/gvla-data'])
subprocess.run(['ls', '-R', '/kaggle/working/gvla-data'])

## Next step

1. **Save Version** → **Quick Save** so `/kaggle/working/gvla-data` becomes a Kaggle
   Dataset (name it `gvla-data`).
2. From here on, mount both `gvla-weights` and `gvla-data` in every eval notebook.

If you ever want a larger evaluation slice, increase `--limit` and re-run.